# 🐝 TTC: Insects 95:5 Standard SupCon Baseline (Table 1)
Chạy 350 Epochs Pre-training + 50 Epochs Linear Probing Evaluation.
Tự động đồng bộ lên WandB và tự ngắt GPU khi hoàn tất.

In [ ]:
# 1. Kiểm tra GPU
!nvidia-smi

In [ ]:
# 2. Clone mã nguồn TTC mới nhất
import os
if not os.path.exists('TTC'):
    !git clone https://github.com/XiaoSha59/TTC.git
%cd TTC
!git pull

In [ ]:
# 3. Cài đặt các thư viện cần thiết
!pip install -q 'lightning>=2.0.0' 'hydra-core>=1.3.2' omegaconf pyrootutils timm

In [ ]:
# 4. Cấu hình WandB API
import os, wandb
os.environ['WANDB_API_KEY'] = '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'
!wandb login $WANDB_API_KEY

In [ ]:
# 5. Liên kết Dataset iNat21 Natural
import os, subprocess, shutil
os.makedirs('data', exist_ok=True)
inat_input = '/kaggle/input/inat21-natural'
if not os.path.exists('data/inat21'):
    if os.path.exists(inat_input):
        os.symlink(inat_input, 'data/inat21')
        print('>>> Đã liên kết dataset từ /kaggle/input/inat21-natural thành công!')
    else:
        print('⚠️ Cảnh báo: Tìm kiếm dataset inat21...')
        !find /kaggle/input -maxdepth 3 -type d

In [ ]:
# 6. [GIAI ĐOẠN 1] Pre-training 350 Epochs Standard SupCon
!python train.py \
    experiment=contrastive \
    experiment/specs=insects \
    class_ratios=[0.05,0.95] \
    module.ratio_supervised_majority=1.0 \
    batch_size=256 \
    trainer.max_epochs=350 \
    module.lr=0.0625 \
    trainer.precision=16-mixed \
    data.data_module.num_workers=2 \
    data.data_module.persistent_workers=False \
    trainer.check_val_every_n_epoch=5 \
    name='insects-95_5-supcon-350ep-full'

In [ ]:
# 7. Tìm checkpoint backbone vừa hoàn tất
import glob
ckpts = sorted(glob.glob('logs/train/runs/*/checkpoints/last.ckpt'), key=os.path.getmtime)
if not ckpts:
    raise FileNotFoundError('Không tìm thấy last.ckpt!')
last_ckpt = ckpts[-1]
print(f'>>> Checkpoint backbone: {last_ckpt}')

# [GIAI ĐOẠN 2] Linear Probing 50 Epochs
!python train.py \
    experiment=finetune \
    experiment/specs=insects \
    +base_model_path={last_ckpt} \
    trainer.max_epochs=50 \
    module.optimizer_name=adam \
    module.lr=0.001 \
    train_transform._target_=data.augmentation.SimCLRValTransform \
    data.data_module.num_workers=2 \
    data.data_module.persistent_workers=False \
    name='insects-95_5-supcon-full-probe'

In [ ]:
print('🎉 Hoàn thành xuất sắc toàn bộ Pipeline Insects Standard SupCon 95:5! Kaggle GPU tự động giải phóng.')